# 5D Cournot deterministic/stochastic analysis through DTB Ver3

Set one option below to run deterministic DTB with an Euler reference or stochastic probability-flow DTB with an Euler--Maruyama reference. `experiment.py` owns the complete run, uses a fixed tangent-coordinate basis, and advances the accumulated particles directly.

[Open in Colab](https://colab.research.google.com/github/sun-mengwei/dtb-colab-experiments/blob/codex%2Fgame-dynamics-dtb/DTB_Ver3/notebooks/cournot_5d_analysis.ipynb)


In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt

# Find a local checkout. In Colab, obtain the same branch automatically.
candidates = [Path.cwd(), *Path.cwd().parents]
repo_root = next((path for path in candidates if (path / 'DTB_Ver3').is_dir()), None)
if repo_root is None:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Run inside the repository or upload the DTB_Ver3 folder.')
    repo_root = Path('/content/dtb-colab-experiments')
    if not repo_root.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1', '--branch', 'codex/game-dynamics-dtb',
            'https://github.com/sun-mengwei/dtb-colab-experiments.git', str(repo_root),
        ], check=True)
sys.path.insert(0, str(repo_root.resolve()))

from DTB_Ver3 import ConstantDiffusion, CournotGame, ExperimentConfig, run_experiment
from DTB_Ver3.utils import (
    plot_cloud_snapshots, plot_diagnostics, plot_stochastic_diagnostics,
)

package_dir = repo_root / 'DTB_Ver3'


## Game, diffusion, and numerical setup

For $s_i=\sum_{j\ne i}x_j$, the drift is

$$
b_i(x)=2b\left(\max\{\mu s_i(1-s_i),0\}-x_i\right),
\qquad b=2,\quad \mu=\frac74.
$$

When `RUN_STOCHASTIC=True`, the SDE adds $\Sigma\,dW_t$ with $\Sigma=0.1I_5$. The probability-flow DTB target includes its score correction, and the reference automatically becomes Euler--Maruyama. The remaining settings preserve the current comparison.


In [ ]:
RUN_STOCHASTIC = False
NOISE_AMPLITUDE = 0.1

game = CournotGame(dim=5, b=2.0, mu=7/4)
diffusion = (
    ConstantDiffusion.isotropic(game.dim, amplitude=NOISE_AMPLITUDE)
    if RUN_STOCHASTIC
    else None
)
mode = 'stochastic' if RUN_STOCHASTIC else 'deterministic'

config = ExperimentConfig(
    dynamics=mode,
    run_reference=True,
    particle_count=3000,
    initial_law='smoothed_uniform',
    smoothing_std=0.02,
    step_size=0.005,
    final_time=2.0,
    snapshot_times=(0.0, 0.5, 1.0, 2.0),
    width=16,
    depth=2,
    activation='tanh',
    basis_size=64,
    svd_rtol=1e-6,
    jacobian_chunk=256,
    score_chunk=32,
    seed=0,
    dtype='float64',
    device='auto',
    progress_reports=5,
    output_dir=package_dir / 'results' / f'cournot_5d_{mode}',
)


## Complete run

This call initializes the cloud and MLP, fixes the selected tangent coordinates, runs all DTB steps, chooses Euler or Euler--Maruyama from `dynamics`, reports progress every 20%, and saves the numerical results.


In [ ]:
result = run_experiment(game, config, diffusion=diffusion)
result.summary()


## Particle clouds and diagnostics


In [ ]:
pairs = ((1, 2), (2, 3), (3, 4), (4, 5))

plot_cloud_snapshots(
    result.dtb_snapshots,
    coordinate_pairs=pairs,
    title=f'5D Cournot {result.dynamics} DTB',
    color='#176b87',
    output_path=result.output_dir / 'dtb_point_clouds.png',
)
plt.show()

if result.reference_snapshots:
    reference_label = (
        'Euler--Maruyama' if result.reference_method == 'euler_maruyama' else 'Euler'
    )
    plot_cloud_snapshots(
        result.reference_snapshots,
        coordinate_pairs=pairs,
        title=f'5D Cournot {reference_label} reference',
        color='#b45309',
        output_path=result.output_dir / 'reference_point_clouds.png',
    )
    plt.show()

plot_diagnostics(
    result.projection_times,
    result.projection_error,
    result.jacobian_condition,
    output_path=result.output_dir / 'dtb_diagnostics.png',
)
plt.show()

if result.dynamics == 'stochastic':
    plot_stochastic_diagnostics(
        result.projection_times,
        result.score_rms,
        result.diffusion_correction_rms,
        output_path=result.output_dir / 'stochastic_diagnostics.png',
    )
    plt.show()


## Package the saved outputs


In [ ]:
import shutil

archive = shutil.make_archive(
    str(result.output_dir),
    'zip',
    root_dir=result.output_dir.parent,
    base_dir=result.output_dir.name,
)
print('Saved result archive:', archive)

try:
    from google.colab import files
except ImportError:
    pass
else:
    files.download(archive)
